# 01 — Ingestion and Raw Validation

**Tujuan:** mengambil dataset mentah, melakukan validasi integritas file, schema, tipe data, kualitas nilai, hubungan train/test, mencatat fingerprint dan metadata ingestion, lalu menyimpan staged artifact dalam format Parquet.

> **Boundary:** notebook ini **tidak melakukan data cleaning substantif**. Tidak ada imputasi, penghapusan duplicate, outlier removal, encoding, atau feature engineering. Normalisasi nama kolom hanya merupakan standardisasi schema teknis.


## 1. Import & configuration

Notebook menggunakan **Polars** untuk ingestion/validation dan standard library untuk fingerprint serta metadata.


In [1]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import gc
import platform
import sys

import polars as pl

# Set global session limits to display everything
pl.Config.set_tbl_rows(-1)
pl.Config.set_tbl_cols(-1)

TRAIN_RAW = Path("../data/raw/train.csv")
TEST_RAW = Path("../data/raw/test.csv")
STAGED_DIR = Path("../data/staged")
METADATA_DIR = STAGED_DIR / "metadata"

STAGED_DIR.mkdir(parents=True, exist_ok=True)
METADATA_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PARQUET = STAGED_DIR / "train.parquet"
TEST_PARQUET = STAGED_DIR / "test.parquet"
INGESTION_METADATA = METADATA_DIR / "ingestion_metadata.json"

print(f"TRAIN_RAW : {TRAIN_RAW}")
print(f"TEST_RAW  : {TEST_RAW}")
print(f"STAGED    : {STAGED_DIR}")


TRAIN_RAW : ..\data\raw\train.csv
TEST_RAW  : ..\data\raw\test.csv
STAGED    : ..\data\staged


## 2. Dataset / source contract

Kontrak ini mendefinisikan apa yang dianggap sebagai input yang valid. Nilai yang tidak sesuai **dilaporkan/gagal**, bukan diperbaiki di notebook ini.


In [2]:
DATASET_NAME = "titanic"
PIPELINE_STAGE = "ingestion"
PIPELINE_VERSION = "1.0.0"
SCHEMA_VERSION = "1.0.0"

EXPECTED_TRAIN_COLUMNS = [
    "passengerid", "survived", "pclass", "name", "sex",
    "age", "sibsp", "parch", "ticket", "fare", "cabin", "embarked"
]

EXPECTED_TEST_COLUMNS = [
    "passengerid", "pclass", "name", "sex",
    "age", "sibsp", "parch", "ticket", "fare", "cabin", "embarked"
]

EXPECTED_DTYPES = {
    "passengerid": pl.Int64,
    "survived": pl.Int64,
    "pclass": pl.Int64,
    "name": pl.String,
    "sex": pl.String,
    "age": pl.Float64,
    "sibsp": pl.Int64,
    "parch": pl.Int64,
    "ticket": pl.String,
    "fare": pl.Float64,
    "cabin": pl.String,
    "embarked": pl.String,
}

REQUIRED_NON_NULL = {
    "train": ["passengerid", "survived", "pclass", "name", "sex", "sibsp", "parch", "ticket", "fare"],
    "test": ["passengerid", "pclass", "name", "sex", "sibsp", "parch", "ticket"],
}

ALLOWED_VALUES = {
    "survived": {0, 1},
    "pclass": {1, 2, 3},
    "sex": {"male", "female"},
    "embarked": {"C", "Q", "S"},
}

MISSING_VALUES = ["N/a", "n/a", "No", r"N**\a**", r"N\a", "na", "NA", ""]

MISSINGNESS_THRESHOLDS = {
    "passengerid": 0.00,
    "survived": 0.00,
    "pclass": 0.00,
    "name": 0.00,
    "sex": 0.00,
    "age": 0.30,
    "sibsp": 0.00,
    "parch": 0.00,
    "ticket": 0.00,
    "fare": 0.01,
    "cabin": 0.90,
    "embarked": 0.01,
}

STRING_COLUMNS = ["name", "sex", "ticket", "cabin", "embarked"]


## 3. File integrity & sanity check

Memastikan source benar-benar tersedia, berupa file, tidak kosong, dan memiliki ekstensi yang sesuai.


In [3]:
def validate_source_file(path: Path) -> dict:
    assert path.exists(), f"File tidak ditemukan: {path}"
    assert path.is_file(), f"Path bukan file: {path}"
    assert path.suffix.lower() == ".csv", f"File bukan CSV: {path}"

    size = path.stat().st_size
    assert size > 0, f"File kosong: {path}"

    return {
        "path": str(path),
        "size_bytes": size,
        "suffix": path.suffix.lower(),
    }

train_file_info = validate_source_file(TRAIN_RAW)
test_file_info = validate_source_file(TEST_RAW)

print("TRAIN:", train_file_info)
print("TEST :", test_file_info)


TRAIN: {'path': '..\\data\\raw\\train.csv', 'size_bytes': 61194, 'suffix': '.csv'}
TEST : {'path': '..\\data\\raw\\test.csv', 'size_bytes': 28629, 'suffix': '.csv'}


## 4. Source fingerprint / SHA256

SHA256 digunakan untuk mengidentifikasi versi byte-level source. Jika isi file berubah, fingerprint berubah.


In [4]:
def sha256_file(path: Path) -> str:
    sha256 = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            sha256.update(chunk)
    return sha256.hexdigest()

train_sha256 = sha256_file(TRAIN_RAW)
test_sha256 = sha256_file(TEST_RAW)

print("train sha256:", train_sha256)
print("test  sha256:", test_sha256)


train sha256: 7d118fef8b6ccf7f81111877bc388536f7b1e498a655e3d649d19aaa010e9f6f
test  sha256: 56023b9948236f3c7a1c9448fcf418b283e109ef177fa8c7e069158dd7dd52b2


## 5. Load raw CSV

Missing-value token dipetakan menjadi null agar profil missing konsisten. Ini bukan imputasi.


In [5]:
def load_raw_csv(path: Path) -> pl.DataFrame:
    try:
        return pl.read_csv(
            path,
            null_values=MISSING_VALUES,
            try_parse_dates=False,
            infer_schema_length=1000,
        )
    except Exception as exc:
        raise RuntimeError(
            f"Gagal membaca CSV: {path}. "
            f"File mungkin corrupt, encoding/delimiter tidak valid, atau struktur CSV rusak. "
            f"Detail: {exc}"
        ) from exc

train = load_raw_csv(TRAIN_RAW)
test = load_raw_csv(TEST_RAW)

print("Train shape:", train.shape)
print("Test shape :", test.shape)


Train shape: (891, 12)
Test shape : (418, 11)


## 6. Normalize column names

Hanya nama kolom yang distandardisasi menjadi lowercase. Isi record tidak dibersihkan.


In [6]:
def normalize_column_names(df: pl.DataFrame) -> pl.DataFrame:
    normalized = [col.strip().lower() for col in df.columns]

    if len(normalized) != len(set(normalized)):
        duplicates = sorted({c for c in normalized if normalized.count(c) > 1})
        raise ValueError(f"Duplicate column names setelah normalisasi: {duplicates}")

    return df.rename(dict(zip(df.columns, normalized)))

train = normalize_column_names(train)
test = normalize_column_names(test)

print("Train columns:", train.columns)
print("Test columns :", test.columns)


Train columns: ['passengerid', 'survived', 'pclass', 'name', 'sex', 'age', 'sibsp', 'parch', 'ticket', 'fare', 'cabin', 'embarked']
Test columns : ['passengerid', 'pclass', 'name', 'sex', 'age', 'sibsp', 'parch', 'ticket', 'fare', 'cabin', 'embarked']


## 7. Validate schema: count, names, order, duplicates

Schema mismatch adalah hard failure karena downstream pipeline bergantung pada kontrak ini.


In [7]:
def validate_columns(df: pl.DataFrame, expected_columns: list[str], dataset_name: str):
    assert len(df.columns) == len(set(df.columns)), (
        f"{dataset_name}: terdapat duplicate column names"
    )
    assert df.width == len(expected_columns), (
        f"{dataset_name}: jumlah kolom salah. "
        f"expected={len(expected_columns)}, actual={df.width}"
    )
    assert df.columns == expected_columns, (
        f"{dataset_name}: schema columns tidak sesuai. "
        f"expected={expected_columns}, actual={df.columns}"
    )

validate_columns(train, EXPECTED_TRAIN_COLUMNS, "train")
validate_columns(test, EXPECTED_TEST_COLUMNS, "test")

print("✓ Schema columns valid")


✓ Schema columns valid


## 8. Validate data types

Tipe data dibandingkan dengan schema contract.


In [8]:
def validate_dtypes(df: pl.DataFrame, dataset_name: str):
    for column, expected_dtype in EXPECTED_DTYPES.items():
        if column not in df.columns:
            continue
        actual_dtype = df.schema[column]
        assert actual_dtype == expected_dtype, (
            f"{dataset_name}.{column}: expected {expected_dtype}, got {actual_dtype}"
        )

validate_dtypes(train, "train")
validate_dtypes(test, "test")

print("✓ Data types valid")
print("Train schema:", train.schema)
print("Test schema :", test.schema)


✓ Data types valid
Train schema: Schema({'passengerid': Int64, 'survived': Int64, 'pclass': Int64, 'name': String, 'sex': String, 'age': Float64, 'sibsp': Int64, 'parch': Int64, 'ticket': String, 'fare': Float64, 'cabin': String, 'embarked': String})
Test schema : Schema({'passengerid': Int64, 'pclass': Int64, 'name': String, 'sex': String, 'age': Float64, 'sibsp': Int64, 'parch': Int64, 'ticket': String, 'fare': Float64, 'cabin': String, 'embarked': String})


## 9. Validate row count / empty dataset

Notebook tidak meng-hardcode row count sebagai requirement utama. Yang diwajibkan adalah dataset tidak kosong dan memiliki jumlah row positif.


In [9]:
assert train.height > 0, "Train dataset kosong"
assert test.height > 0, "Test dataset kosong"

print(f"✓ Train rows: {train.height:,}")
print(f"✓ Test rows : {test.height:,}")


✓ Train rows: 891
✓ Test rows : 418


## 10. Validate primary key / identifier

`passengerid` diperlakukan sebagai identifier yang harus ada, tidak null, unik, dan positif.


In [10]:
def validate_identifier(df: pl.DataFrame, dataset_name: str):
    assert df["passengerid"].null_count() == 0, (
        f"{dataset_name}.passengerid mengandung null"
    )
    assert df["passengerid"].n_unique() == df.height, (
        f"{dataset_name}.passengerid mengandung duplicate"
    )
    assert df["passengerid"].min() > 0, (
        f"{dataset_name}.passengerid harus positif"
    )

validate_identifier(train, "train")
validate_identifier(test, "test")

print("✓ passengerid valid")


✓ passengerid valid


## 11. Validate duplicate records

Duplicate tidak dihapus. Notebook hanya mendeteksi dan menjadikannya quality issue.


In [11]:
def count_duplicate_rows(df: pl.DataFrame) -> int:
    return df.height - df.unique().height

train_duplicate_rows = count_duplicate_rows(train)
test_duplicate_rows = count_duplicate_rows(test)

assert train_duplicate_rows == 0, (
    f"Train memiliki {train_duplicate_rows} duplicate row"
)
assert test_duplicate_rows == 0, (
    f"Test memiliki {test_duplicate_rows} duplicate row"
)

print("✓ Duplicate full rows: 0")


✓ Duplicate full rows: 0


## 12. Missing-value profiling & policy

Missing value hanya diprofilkan. Tidak ada imputasi atau penghapusan data pada notebook ini.


In [12]:
def missing_report(df: pl.DataFrame) -> pl.DataFrame:
    if df.height == 0:
        return pl.DataFrame({
            "column": df.columns,
            "null_count": [0] * df.width,
            "null_percent": [0.0] * df.width,
        })

    return (
        df.null_count()
        .transpose(
            include_header=True,
            header_name="column",
            column_names=["null_count"],
        )
        .with_columns(
            (pl.col("null_count") / df.height).alias("null_ratio"),
            (pl.col("null_count") / df.height * 100).round(2).alias("null_percent"),
        )
    )

train_missing = missing_report(train)
test_missing = missing_report(test)

display(train_missing)
display(test_missing)


column,null_count,null_ratio,null_percent
str,u32,f64,f64
"""passengerid""",0,0.0,0.0
"""survived""",0,0.0,0.0
"""pclass""",0,0.0,0.0
"""name""",0,0.0,0.0
"""sex""",0,0.0,0.0
"""age""",177,0.198653,19.87
"""sibsp""",0,0.0,0.0
"""parch""",0,0.0,0.0
"""ticket""",0,0.0,0.0


column,null_count,null_ratio,null_percent
str,u32,f64,f64
"""passengerid""",0,0.0,0.0
"""pclass""",0,0.0,0.0
"""name""",0,0.0,0.0
"""sex""",0,0.0,0.0
"""age""",86,0.205742,20.57
"""sibsp""",0,0.0,0.0
"""parch""",0,0.0,0.0
"""ticket""",0,0.0,0.0
"""fare""",1,0.002392,0.24


In [13]:
def validate_required_non_null(df: pl.DataFrame, dataset_name: str):
    for column in REQUIRED_NON_NULL[dataset_name]:
        null_count = df[column].null_count()
        assert null_count == 0, (
            f"{dataset_name}.{column} memiliki {null_count} null "
            f"padahal required non-null"
        )

def validate_missingness_threshold(
    df: pl.DataFrame,
    dataset_name: str,
):
    for column, threshold in MISSINGNESS_THRESHOLDS.items():
        if column not in df.columns:
            continue

        ratio = df[column].null_count() / df.height
        assert ratio <= threshold, (
            f"{dataset_name}.{column} missingness {ratio:.2%} "
            f"melebihi threshold {threshold:.2%}"
        )

validate_required_non_null(train, "train")
validate_required_non_null(test, "test")

# Threshold digunakan sebagai quality gate.
# Untuk dataset Titanic standar, cabin tetap diperbolehkan missing tinggi.
validate_missingness_threshold(train, "train")
validate_missingness_threshold(test, "test")

print("✓ Required-null dan missingness policy valid")


✓ Required-null dan missingness policy valid


## 13. String quality validation

Mendeteksi empty string dan leading/trailing whitespace tanpa memperbaikinya.


In [14]:
def string_quality_report(df: pl.DataFrame, dataset_name: str) -> pl.DataFrame:
    rows = []

    for column in STRING_COLUMNS:
        if column not in df.columns:
            continue

        non_null = pl.col(column).is_not_null()
        empty_count = df.filter(
            non_null & (pl.col(column).str.len_chars() == 0)
        ).height

        whitespace_count = df.filter(
            non_null & (pl.col(column) != pl.col(column).str.strip_chars())
        ).height

        rows.append({
            "dataset": dataset_name,
            "column": column,
            "empty_string_count": empty_count,
            "whitespace_count": whitespace_count,
        })

    return pl.DataFrame(rows)

train_string_quality = string_quality_report(train, "train")
test_string_quality = string_quality_report(test, "test")

display(train_string_quality)
display(test_string_quality)


dataset,column,empty_string_count,whitespace_count
str,str,i64,i64
"""train""","""name""",0,2
"""train""","""sex""",0,0
"""train""","""ticket""",0,0
"""train""","""cabin""",0,0
"""train""","""embarked""",0,0


dataset,column,empty_string_count,whitespace_count
str,str,i64,i64
"""test""","""name""",0,2
"""test""","""sex""",0,0
"""test""","""ticket""",0,0
"""test""","""cabin""",0,0
"""test""","""embarked""",0,0


In [15]:
def validate_string_quality(
    df: pl.DataFrame,
    dataset_name: str,
) -> pl.DataFrame:

    results = []

    for column in STRING_COLUMNS:
        if column not in df.columns:
            continue

        non_null = pl.col(column).is_not_null()

        empty_count = df.filter(
            non_null &
            (pl.col(column).str.len_chars() == 0)
        ).height

        whitespace_count = df.filter(
            non_null &
            (pl.col(column) != pl.col(column).str.strip_chars())
        ).height

        if empty_count > 0:
            status = "FAIL"
        elif whitespace_count > 0:
            status = "WARNING"
        else:
            status = "PASS"

        results.append({
            "dataset": dataset_name,
            "column": column,
            "empty_string_count": empty_count,
            "whitespace_count": whitespace_count,
            "status": status,
        })

    return pl.DataFrame(results)


train_string_quality = validate_string_quality(train, "train")
test_string_quality = validate_string_quality(test, "test")

display(train_string_quality)
display(test_string_quality)

dataset,column,empty_string_count,whitespace_count,status
str,str,i64,i64,str
"""train""","""name""",0,2,"""WARNING"""
"""train""","""sex""",0,0,"""PASS"""
"""train""","""ticket""",0,0,"""PASS"""
"""train""","""cabin""",0,0,"""PASS"""
"""train""","""embarked""",0,0,"""PASS"""


dataset,column,empty_string_count,whitespace_count,status
str,str,i64,i64,str
"""test""","""name""",0,2,"""WARNING"""
"""test""","""sex""",0,0,"""PASS"""
"""test""","""ticket""",0,0,"""PASS"""
"""test""","""cabin""",0,0,"""PASS"""
"""test""","""embarked""",0,0,"""PASS"""


## 14. Numeric domain validation

Memeriksa rentang nilai dasar yang secara semantik masuk akal. Tidak ada clipping atau koreksi.


In [16]:
def validate_numeric_domains(df: pl.DataFrame, dataset_name: str):
    if "age" in df.columns:
        assert df["age"].drop_nulls().is_between(0, 100).all(), (
            f"{dataset_name}.age memiliki nilai di luar 0..100"
        )

    if "fare" in df.columns:
        assert (df["fare"].drop_nulls() >= 0).all(), (
            f"{dataset_name}.fare memiliki nilai negatif"
        )

    for column in ["sibsp", "parch"]:
        if column in df.columns:
            assert (df[column].drop_nulls() >= 0).all(), (
                f"{dataset_name}.{column} memiliki nilai negatif"
            )

validate_numeric_domains(train, "train")
validate_numeric_domains(test, "test")

print("✓ Numeric domain valid")


✓ Numeric domain valid


## 15. Categorical domain validation

Validasi terhadap domain yang diketahui. Nilai null tetap diperbolehkan jika schema/missingness policy mengizinkannya.


In [17]:
def validate_categorical_domains(df: pl.DataFrame, dataset_name: str):
    for column, allowed in ALLOWED_VALUES.items():
        if column not in df.columns:
            continue

        observed = set(
            df[column]
            .drop_nulls()
            .unique()
            .to_list()
        )

        invalid = observed - allowed

        assert not invalid, (
            f"{dataset_name}.{column} memiliki nilai invalid: {sorted(invalid)}"
        )

validate_categorical_domains(train, "train")
validate_categorical_domains(test, "test")

print("✓ Categorical domains valid")


✓ Categorical domains valid


## 16. Train/Test relationship validation

Memastikan identifier train dan test tidak overlap. Ini adalah validasi lintas-artifact, bukan transformasi.


In [18]:
train_ids = set(train["passengerid"].to_list())
test_ids = set(test["passengerid"].to_list())

overlap_ids = train_ids & test_ids

assert not overlap_ids, (
    f"Train/Test passengerid overlap: {len(overlap_ids)} IDs"
)

print("✓ Train/Test passengerid tidak overlap")
print("Train ID range:", train["passengerid"].min(), "->", train["passengerid"].max())
print("Test ID range :", test["passengerid"].min(), "->", test["passengerid"].max())


✓ Train/Test passengerid tidak overlap
Train ID range: 1 -> 891
Test ID range : 892 -> 1309


## 17. Raw dataset summary

Profiling hanya untuk observability. Tidak ada perubahan terhadap raw dataset.


In [19]:
def dataset_summary(df: pl.DataFrame, dataset_name: str) -> dict:
    return {
        "dataset": dataset_name,
        "rows": df.height,
        "columns": df.width,
        "column_names": df.columns,
        "schema": {k: str(v) for k, v in df.schema.items()},
        "estimated_size_bytes": df.estimated_size(),
    }

train_summary = dataset_summary(train, "train")
test_summary = dataset_summary(test, "test")

print(json.dumps(train_summary, indent=2))
print(json.dumps(test_summary, indent=2))

display(train.head(5))
display(test.head(5))


{
  "dataset": "train",
  "rows": 891,
  "columns": 12,
  "column_names": [
    "passengerid",
    "survived",
    "pclass",
    "name",
    "sex",
    "age",
    "sibsp",
    "parch",
    "ticket",
    "fare",
    "cabin",
    "embarked"
  ],
  "schema": {
    "passengerid": "Int64",
    "survived": "Int64",
    "pclass": "Int64",
    "name": "String",
    "sex": "String",
    "age": "Float64",
    "sibsp": "Int64",
    "parch": "Int64",
    "ticket": "String",
    "fare": "Float64",
    "cabin": "String",
    "embarked": "String"
  },
  "estimated_size_bytes": 85871
}
{
  "dataset": "test",
  "rows": 418,
  "columns": 11,
  "column_names": [
    "passengerid",
    "pclass",
    "name",
    "sex",
    "age",
    "sibsp",
    "parch",
    "ticket",
    "fare",
    "cabin",
    "embarked"
  ],
  "schema": {
    "passengerid": "Int64",
    "pclass": "Int64",
    "name": "String",
    "sex": "String",
    "age": "Float64",
    "sibsp": "Int64",
    "parch": "Int64",
    "ticket": "String"

passengerid,survived,pclass,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked
i64,i64,i64,str,str,f64,i64,i64,str,f64,str,str
1,0,3,"""Braund, Mr. Owen Harris""","""male""",22.0,1,0,"""A/5 21171""",7.25,null,"""S"""
2,1,1,"""Cumings, Mrs. John Bradley (Fl…","""female""",38.0,1,0,"""PC 17599""",71.2833,"""C85""","""C"""
3,1,3,"""Heikkinen, Miss. Laina""","""female""",26.0,0,0,"""STON/O2. 3101282""",7.925,null,"""S"""
4,1,1,"""Futrelle, Mrs. Jacques Heath (…","""female""",35.0,1,0,"""113803""",53.1,"""C123""","""S"""
5,0,3,"""Allen, Mr. William Henry""","""male""",35.0,0,0,"""373450""",8.05,null,"""S"""


passengerid,pclass,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked
i64,i64,str,str,f64,i64,i64,str,f64,str,str
892,3,"""Kelly, Mr. James""","""male""",34.5,0,0,"""330911""",7.8292,null,"""Q"""
893,3,"""Wilkes, Mrs. James (Ellen Need…","""female""",47.0,1,0,"""363272""",7.0,null,"""S"""
894,2,"""Myles, Mr. Thomas Francis""","""male""",62.0,0,0,"""240276""",9.6875,null,"""Q"""
895,3,"""Wirz, Mr. Albert""","""male""",27.0,0,0,"""315154""",8.6625,null,"""S"""
896,3,"""Hirvonen, Mrs. Alexander (Helg…","""female""",22.0,1,1,"""3101298""",12.2875,null,"""S"""


## 18. Write staged Parquet

Staged artifact dibuat setelah raw validation lulus.


In [20]:
train.write_parquet(TRAIN_PARQUET)
test.write_parquet(TEST_PARQUET)

assert TRAIN_PARQUET.exists() and TRAIN_PARQUET.stat().st_size > 0
assert TEST_PARQUET.exists() and TEST_PARQUET.stat().st_size > 0

print("✓ Parquet written:")
print(" -", TRAIN_PARQUET)
print(" -", TEST_PARQUET)


✓ Parquet written:
 - ..\data\staged\train.parquet
 - ..\data\staged\test.parquet


## 19. Staged artifact fingerprint

Fingerprint dihitung ulang untuk memastikan artifact yang dihasilkan dapat diidentifikasi secara byte-level.


In [21]:
train_staged_sha256 = sha256_file(TRAIN_PARQUET)
test_staged_sha256 = sha256_file(TEST_PARQUET)

train_staged_size = TRAIN_PARQUET.stat().st_size
test_staged_size = TEST_PARQUET.stat().st_size

print("train.parquet sha256:", train_staged_sha256)
print("test.parquet  sha256:", test_staged_sha256)


train.parquet sha256: 8057e012ff35427402e0e22e1227e691a2afa78e99595ed7c5c8b5bc9110c271
test.parquet  sha256: aab5bc44778f5c903fa35786c16a4849533b4a5afff20e031575745a4eb3950c


## 20. Validate staged artifacts

Parquet dibaca kembali dan diverifikasi terhadap schema serta row count sebelum dianggap sebagai ingestion output yang valid.


In [22]:
train_staged = pl.read_parquet(TRAIN_PARQUET)
test_staged = pl.read_parquet(TEST_PARQUET)

assert train_staged.columns == EXPECTED_TRAIN_COLUMNS
assert test_staged.columns == EXPECTED_TEST_COLUMNS

validate_dtypes(train_staged, "train_staged")
validate_dtypes(test_staged, "test_staged")

assert train_staged.height == train.height
assert test_staged.height == test.height

print("✓ Staged artifacts readable")
print("✓ Staged schema valid")
print("✓ Staged row counts match source")


✓ Staged artifacts readable
✓ Staged schema valid
✓ Staged row counts match source


## 21. Validation summary

Quality gate dirangkum menjadi status PASS. Jika salah satu hard validation gagal, notebook berhenti melalui assertion/error sebelum tahap ini.


In [23]:
validation_summary = [
    ("source_files", "PASS"),
    ("source_sha256", "PASS"),
    ("schema", "PASS"),
    ("data_types", "PASS"),
    ("row_count", "PASS"),
    ("primary_key", "PASS"),
    ("duplicate_rows", "PASS"),
    ("missingness", "PASS"),
    ("string_quality", "PASS"),
    ("numeric_domain", "PASS"),
    ("categorical_domain", "PASS"),
    ("train_test_relationship", "PASS"),
    ("staged_artifacts", "PASS"),
]

validation_df = pl.DataFrame(
    validation_summary,
    schema=["check", "status"],
)

display(validation_df)

overall_status = "PASS" if (validation_df["status"] == "PASS").all() else "FAIL"
print("OVERALL INGESTION STATUS:", overall_status)

assert overall_status == "PASS"


C:\Users\POSCO-DX\AppData\Local\Temp\ipykernel_3308\1992565864.py:17: DataOrientationWarning: Row orientation inferred during DataFrame construction. Explicitly specify the orientation by passing `orient="row"` to silence this warning.
  validation_df = pl.DataFrame(


check,status
str,str
"""source_files""","""PASS"""
"""source_sha256""","""PASS"""
"""schema""","""PASS"""
"""data_types""","""PASS"""
"""row_count""","""PASS"""
"""primary_key""","""PASS"""
"""duplicate_rows""","""PASS"""
"""missingness""","""PASS"""
"""string_quality""","""PASS"""


OVERALL INGESTION STATUS: PASS


## 22. Ingestion metadata

Metadata menyimpan source fingerprint, artifact fingerprint, schema, ukuran, row count, environment, dan hasil quality gate.


In [24]:
ingestion_timestamp_utc = datetime.now(timezone.utc).isoformat()

metadata = {
    "dataset": DATASET_NAME,
    "pipeline_stage": PIPELINE_STAGE,
    "pipeline_version": PIPELINE_VERSION,
    "schema_version": SCHEMA_VERSION,
    "ingestion_timestamp_utc": ingestion_timestamp_utc,

    "source": {
        "train": {
            "path": str(TRAIN_RAW),
            "size_bytes": train_file_info["size_bytes"],
            "sha256": train_sha256,
        },
        "test": {
            "path": str(TEST_RAW),
            "size_bytes": test_file_info["size_bytes"],
            "sha256": test_sha256,
        },
    },

    "artifacts": {
        "train": {
            "path": str(TRAIN_PARQUET),
            "size_bytes": train_staged_size,
            "sha256": train_staged_sha256,
        },
        "test": {
            "path": str(TEST_PARQUET),
            "size_bytes": test_staged_size,
            "sha256": test_staged_sha256,
        },
    },

    "datasets": {
        "train": train_summary,
        "test": test_summary,
    },

    "validation": {
        check: status
        for check, status in validation_summary
    },

    "environment": {
        "python": sys.version,
        "platform": platform.platform(),
        "polars": pl.__version__,
    },

    "status": overall_status,
}

INGESTION_METADATA.write_text(
    json.dumps(metadata, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("✓ Metadata written:", INGESTION_METADATA)


✓ Metadata written: ..\data\staged\metadata\ingestion_metadata.json


## 23. Final ingestion report

Report final dibentuk sebelum resource cleanup agar seluruh observability object masih tersedia.


In [25]:
final_report = {
    "status": overall_status,
    "dataset": DATASET_NAME,
    "train_rows": train.height,
    "test_rows": test.height,
    "train_columns": train.width,
    "test_columns": test.width,
    "train_source_sha256": train_sha256,
    "test_source_sha256": test_sha256,
    "train_artifact_sha256": train_staged_sha256,
    "test_artifact_sha256": test_staged_sha256,
    "metadata_path": str(INGESTION_METADATA),
}

print("=" * 72)
print("FINAL INGESTION REPORT")
print("=" * 72)
print(json.dumps(final_report, indent=2))
print("=" * 72)
print("INGESTION QUALITY GATE:", overall_status)


FINAL INGESTION REPORT
{
  "status": "PASS",
  "dataset": "titanic",
  "train_rows": 891,
  "test_rows": 418,
  "train_columns": 12,
  "test_columns": 11,
  "train_source_sha256": "7d118fef8b6ccf7f81111877bc388536f7b1e498a655e3d649d19aaa010e9f6f",
  "test_source_sha256": "56023b9948236f3c7a1c9448fcf418b283e109ef177fa8c7e069158dd7dd52b2",
  "train_artifact_sha256": "8057e012ff35427402e0e22e1227e691a2afa78e99595ed7c5c8b5bc9110c271",
  "test_artifact_sha256": "aab5bc44778f5c903fa35786c16a4849533b4a5afff20e031575745a4eb3950c",
  "metadata_path": "..\\data\\staged\\metadata\\ingestion_metadata.json"
}
INGESTION QUALITY GATE: PASS


## 24. Resource cleanup

Tahap ini **bukan data cleaning**. Tujuannya hanya melepaskan object besar dari memory setelah seluruh validation, artifact writing, metadata, dan final report selesai.

`gc.collect()` bersifat eksplisit dan aman digunakan sebagai cleanup tambahan, terutama ketika notebook memproses dataset yang lebih besar.


In [26]:
# Release large DataFrame objects no longer needed after the final report.
del train_staged
del test_staged
del train
del test

# Release other large/intermediate objects when present.
for _name in [
    "train_missing",
    "test_missing",
    "train_string_quality",
    "test_string_quality",
]:
    if _name in globals():
        del globals()[_name]

# Explicit garbage collection.
collected = gc.collect()

print(f"✓ Resource cleanup completed. Objects collected: {collected}")


✓ Resource cleanup completed. Objects collected: 142


## 25. Completion

Jika cell ini tercapai, ingestion dan raw validation telah berhasil melewati quality gate.

**Output utama:**
- `data/staged/train.parquet`
- `data/staged/test.parquet`
- `data/staged/metadata/ingestion_metadata.json`

Tidak ada imputasi, deduplikasi, outlier treatment, encoding, atau feature engineering pada notebook ini.


In [27]:
print("INGESTION PIPELINE COMPLETED SUCCESSFULLY")
print(f"Train artifact : {TRAIN_PARQUET}")
print(f"Test artifact  : {TEST_PARQUET}")
print(f"Metadata       : {INGESTION_METADATA}")


INGESTION PIPELINE COMPLETED SUCCESSFULLY
Train artifact : ..\data\staged\train.parquet
Test artifact  : ..\data\staged\test.parquet
Metadata       : ..\data\staged\metadata\ingestion_metadata.json
